# VAE Face Generator — Colab Training Notebook

## Setup Instructions

1. Go to **Runtime → Change runtime type** and select **T4 GPU** (or any available GPU).
2. Run all cells in order.
3. After training completes, download `vae_model.pth` from the file browser on the left (or use the download cell at the bottom).
4. Place `vae_model.pth` in the same directory as `gui.py` and `vae_model.py` on your local machine, then run `python gui.py`.

## Install Dependencies

PyTorch and NumPy are pre-installed on Colab. We only need scikit-learn for the LFW dataset.

In [ ]:
!pip install -q scikit-learn

## Verify GPU is available

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## VAE Model Definition

In [ ]:
# Correa (last-name, first-name)
# 100x_xxx_xxx
# 2026_03_30
# Assignment_04

import torch
import torch.nn as nn


class Encoder(nn.Module):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # 64 -> 32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 32 -> 16
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 16 -> 8
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), # 8 -> 4
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_log_var = nn.Linear(256 * 4 * 4, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)
        return mu, log_var


class Decoder(nn.Module):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), # 4 -> 8
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 8 -> 16
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),   # 16 -> 32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),    # 32 -> 64
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 256, 4, 4)
        x = self.deconv(x)
        return x


class VAE(nn.Module):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z)
        return recon, mu, log_var


def vae_loss(recon_x, x, mu, log_var):
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + kl_loss

## Load and Preprocess LFW Dataset

In [ ]:
import torch.nn.functional as F
from sklearn.datasets import fetch_lfw_people

IMAGE_SIZE = 64

print("Downloading/loading LFW dataset...")
lfw = fetch_lfw_people(min_faces_per_person=1, resize=1.0, color=True)
images = lfw.images  # (N, H, W, 3), values in [0, 255]
print(f"Loaded {len(images)} images, original shape: {images.shape[1:]}")

# Normalize to [0, 1]
images = images / 255.0 if images.max() > 1.0 else images

# Convert to (N, C, H, W) tensor and resize
data = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)
data = F.interpolate(data, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)
print(f"Resized to {IMAGE_SIZE}x{IMAGE_SIZE}, tensor shape: {data.shape}")

## Train the VAE

In [ ]:
# Correa (last-name, first-name)
# 100x_xxx_xxx
# 2026_03_30
# Assignment_04

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

LATENT_DIM = 6
BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
MODEL_PATH = "vae_model.pth"

dataset = TensorDataset(data)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = VAE(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Training VAE for {EPOCHS} epochs on {device}...\n")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for (batch,) in dataloader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, log_var = model(batch)
        loss = vae_loss(recon, batch, mu, log_var)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(data)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{EPOCHS}] - Avg Loss: {avg_loss:.4f}")

print("\nTraining complete!")

## Save Model and Download

In [ ]:
# Save model (move to CPU for portability)
model = model.cpu()
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

In [ ]:
from google.colab import files
files.download(MODEL_PATH)

## Preview: Sample Reconstructions (Optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.eval()
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    z = torch.randn(1, LATENT_DIM)
    with torch.no_grad():
        img = model.decoder(z).squeeze(0).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[0][i].imshow(img)
    axes[0][i].axis("off")
    axes[0][i].set_title("Generated")

# Show some originals for comparison
for i in range(5):
    img = data[i].permute(1, 2, 0).numpy()
    axes[1][i].imshow(img)
    axes[1][i].axis("off")
    axes[1][i].set_title("Original")

plt.tight_layout()
plt.show()